# GroupDRO++ Notebook (Simple Start)

Notebook này là phiên bản **đơn giản hóa** để triển khai ý tưởng GroupDRO++ từ paper
**"Automated Domain Discovery from Multiple Sources to Improve Zero-Shot Generalization"**.

## Cấu trúc 5 phần (roadmap)
1. **Part 1 (đang làm): Setup + Config + khung Eval tối giản**
2. Part 2: Data pipeline từ nhiều source + split (train/val/test/zero-shot)
3. Part 3: Domain discovery module (prototype) + gán pseudo-domain
4. Part 4: Train GroupDRO++ loop + logging (loss/group-weights/metrics)
5. Part 5: Evaluation đầy đủ (ERM vs GroupDRO vs GroupDRO++) + báo cáo

> Mục tiêu của Part 1: giữ mọi thứ rõ ràng, chạy được, và dễ mở rộng dần theo từng phần.


## Part 1 — Setup môi trường, seed, config và eval schema

Ở phần này chúng ta chỉ dựng **khung xương chuẩn**:
- import package
- set random seed để reproducible
- khai báo config tập trung
- tạo schema cho eval metrics để các phần sau tái sử dụng


In [ ]:
import random
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional

import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {DEVICE}")


### Vì sao cần cell trên?
- `set_seed(...)` giúp kết quả ổn định hơn giữa các lần chạy.
- `DEVICE` giúp notebook tự chọn GPU nếu có.
- Cách tách setup sớm giúp các phần sau chỉ tập trung vào thuật toán.


In [ ]:
@dataclass
class ExperimentConfig:
    # Reproducibility
    seed: int = 42

    # Data / batching
    batch_size: int = 128
    num_workers: int = 2

    # Optimization
    lr: float = 1e-3
    weight_decay: float = 1e-4
    max_epochs: int = 10

    # GroupDRO++ placeholders (sẽ mở rộng ở Part 3-4)
    use_groupdropp: bool = True
    group_update_lr: float = 0.05
    num_discovered_domains: int = 4

    # Eval schema
    eval_splits: List[str] = None
    primary_metric: str = "worst_group_acc"

    def __post_init__(self):
        if self.eval_splits is None:
            self.eval_splits = ["train", "val", "test", "zero_shot"]


CFG = ExperimentConfig()
print(asdict(CFG))


In [ ]:
def init_eval_store(cfg: ExperimentConfig) -> Dict[str, Dict[str, Optional[float]]]:
    # Khởi tạo nơi lưu metrics theo split.
    # Part 4-5 sẽ ghi kết quả thật vào đây.
    metric_template = {
        "avg_acc": None,
        "worst_group_acc": None,
        "avg_loss": None,
    }
    return {split: dict(metric_template) for split in cfg.eval_splits}


eval_store = init_eval_store(CFG)
eval_store


## Kết quả Part 1
Bạn đã có một khung notebook sạch và dễ mở rộng:
- config tập trung (`ExperimentConfig`)
- schema eval thống nhất (`init_eval_store`)
- sẵn sàng nối sang Part 2 (data nhiều nguồn + zero-shot split)

Nếu bạn đồng ý, bước tiếp theo mình sẽ làm **Part 2** theo đúng cấu trúc notebook gốc nhưng giữ code ngắn gọn.
